# HW2-期中作业报告
#### CS50028.01 计算机视觉
#### 洪家权 23307110248@m.fudan.edu.cn

### 1. **任务1**：微调在 ImageNet 上预训练的卷积神经网络实现宠物识别

本任务在 Oxford-IIIT Pet（37 类）上进行宠物品种分类。实验包含四部分：ResNet-18 预训练微调基线、学习率与训练轮数的网格超参数搜索与分析、预训练与随机初始化的消融对比、以及基线模型与引入不同注意力机制后的性能对比。除 **1.5** 外，文中**基线模型**均指同一设置：**ResNet-18、无注意力、ImageNet 预训练微调**；1.5.1 中表 2 涉及到的基线模型，在维持前述基线模型其他配置不变的基础上，将训练轮次延长到 40 代；1.5.2 中单独使用与 Transformer 训练配方对齐的特殊基线。评价指标统一为验证集**最优分类准确率（top-1 Val-Accuracy）**。

#### 1.1 数据集与公共设置说明

数据集采用 Oxford-IIIT Pet 官方图像与品种标签，共 **37** 类；输入分辨率统一为 **224×224**。当前数据划分方式为：先使用官方 `trainval.txt` 作为开发集，并按固定随机种子 **42** 在其中随机划分 **90%** 训练集与 **10%** 验证集；官方 `test.txt` 作为独立测试集，仅用于最终测试指标统计。卷积实验以 **ResNet-18** 为骨干，加载 ImageNet 预训练权重后替换全连接层，并对骨干与新区采用差异化学习率微调。批量大小为 **32**。1.5.2 之外的基线模型训练、预训练消融实验、超参网格搜索以及加入SE\CBAM注意力模块实验中，均使用 **SGD**（动量 0.9，权重衰减 **$10^{-4}$**）和余弦退火学习率调度；仅 **1.5.2** 的 Transformer 对比组及其对照的特殊基线使用 **AdamW** 与带 warmup 的余弦退火学习率调度。全部实验固定随机种子 **42**。

下面对 1.5.2 的训练配方选择，作机制层面的说明。对 Transformer 而言，采用 **AdamW 而非 SGD** 的核心原因在于其对参数尺度与梯度方差更鲁棒。SGD（含动量）更新可写为
$$
v_t=\mu v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta v_t,
$$
其中所有参数共享同一全局学习率 $\eta$，当不同层（尤其是注意力与 MLP 层）梯度统计差异较大时，优化步长难以同时匹配。AdamW 通过一阶/二阶矩估计进行逐参数自适应缩放：
$$
m_t=\beta_1 m_{t-1}+(1-\beta_1)g_t,\quad v_t=\beta_2 v_{t-1}+(1-\beta_2)g_t^2,
$$
$$
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}-\eta\lambda\theta_t,
$$
其中 $-\eta\lambda\theta_t$ 为**解耦权重衰减**项，使正则化强度与梯度自适应缩放分离，通常比在自适应优化器中直接用 $L_2$ 惩罚更稳定。基于这一点，Transformer 组采用 AdamW 以获得更平滑的早期收敛和更好的最终泛化。

在学习率调度上，单纯余弦退火
$$
\eta_t=\eta_{\min}+\frac{1}{2}(\eta_{\max}-\eta_{\min})\left(1+\cos\frac{\pi t}{T}\right)
$$
虽能在中后期实现平滑下降，但若训练初期直接使用较大学习率，Transformer 容易出现梯度波动。因而在其前加入 warmup：
$$
\eta_t=\eta_{\text{start}}+\frac{t}{T_w}(\eta_{\max}-\eta_{\text{start}}),\quad 0\le t<T_w,
$$
随后再切换到余弦退火。该策略的优势是先以小步长稳定表征对齐，再利用余弦下降提升后期收敛与泛化，因此在 Transformer 配方中更合适。

#### 1.2 基线模型

如前所述，基线模型仅替换 Resnet-18 的分类头并进行差异化学习率微调，使用 **SGD**（动量 0.9，权重衰减 **$10^{-4}$**）和余弦退火学习率调度。其配置与结果见下表。

| 指标 | 数值/设置 |
|------|-----------|
| 模型 | ResNet-18（无额外注意力） |
| 训练轮数 | 20 epoch |
| Batch size | 32 |
| 权重衰减 | $10^{-4}$ |
| 骨干学习率 | $10^{-4}$ |
| 分类头学习率 | $10^{-3}$ |
| 验证集最佳准确率 | **0.9271** |
| 测试集准确率 | **0.8914** |
| 最佳 epoch | 18 |
| 末 epoch 训练集准确率 | 0.8450 |
| 末 epoch 验证集准确率 | 0.9167 |

下图为基线模型的训练集与验证集loss/accuracy曲线。

![](task1_classification/ReportFig/1_2.png)

结合图 1.2 可见，20 个 epoch 内训练与验证 loss 均持续下降，其中验证 loss 在末轮仍降至最低；accuracy 曲线整体上升并在第 18 代达到验证集峰值 **92.71%**，末轮验证准确率为 **91.67%**，较峰值仅小幅回落，说明该基线在当前训练轮次下收敛较充分且泛化表现稳定。

需要说明的是，容易注意到，曲线中出现验证准确率长期高于训练准确率：这是因为训练端采用 `RandomResizedCrop` 与 ` RandomHorizontalFlip` 等更强数据增强，样本难度高于验证端 `Resize` 配合 `CenterCrop`。因此在预训练 ResNet-18 的微调场景下，出现该现象是合理的。

#### 1.3 超参数分析

在`「ResNet-18 + ImageNet 预训练 + 无注意力」`的相同设置下，控制batch size与权重衰减不变，本实验对 **backbone_lr(骨干学习率)**、**head_lr(分类头学习率)** 与 **epochs(训练轮数)** 三个超参数进行 **3×3×3** 的超参数网格搜索：三者的取值分别取自 {$10^{-4}$, $2\times10^{-4}$, $3\times10^{-4}$}、{$10^{-3}$, $2\times10^{-3}$, $3\times10^{-3}$} 与 {$20$, $30$, $40$}，共 27 组独立训练。下表按验证集最佳准确率从高到低排序。

| 骨干学习率 | 分类头学习率 | 训练轮数 | 最佳验证集准确率 | 测试集准确率 |
|----------|------|------|---------|-------|
| 0.0003 | 0.003 | 40 | 0.9401 | 0.9062 |
| 0.0001 | 0.003 | 40 | 0.9401 | 0.9047 |
| 0.0003 | 0.003 | 30 | 0.9375 | 0.9119 |
| 0.0002 | 0.003 | 30 | 0.9375 | 0.9092 |
| 0.0002 | 0.002 | 40 | 0.9375 | 0.9090 |
| 0.0002 | 0.003 | 40 | 0.9375 | 0.9084 |
| 0.0001 | 0.003 | 30 | 0.9375 | 0.9065 |
| 0.0003 | 0.002 | 30 | 0.9349 | 0.9106 |
| 0.0003 | 0.002 | 40 | 0.9349 | 0.9095 |
| 0.0002 | 0.002 | 30 | 0.9349 | 0.9082 |
| 0.0001 | 0.003 | 20 | 0.9349 | 0.9034 |
| 0.0001 | 0.002 | 40 | 0.9349 | 0.9033 |
| 0.0003 | 0.001 | 40 | 0.9323 | 0.9101 |
| 0.0003 | 0.001 | 30 | 0.9323 | 0.9090 |
| 0.0002 | 0.001 | 40 | 0.9323 | 0.9061 |
| 0.0001 | 0.002 | 30 | 0.9323 | 0.9053 |
| 0.0003 | 0.003 | 20 | 0.9297 | 0.9083 |
| 0.0002 | 0.003 | 20 | 0.9297 | 0.9060 |
| 0.0002 | 0.001 | 30 | 0.9297 | 0.9053 |
| 0.0001 | 0.001 | 40 | 0.9297 | 0.9007 |
| 0.0003 | 0.002 | 20 | 0.9271 | 0.9105 |
| 0.0002 | 0.002 | 20 | 0.9271 | 0.9034 |
| 0.0003 | 0.001 | 20 | 0.9271 | 0.9031 |
| 0.0002 | 0.001 | 20 | 0.9271 | 0.8990 |
| 0.0001 | 0.001 | 30 | 0.9271 | 0.8966 |
| 0.0001 | 0.001 | 20 | 0.9271 | 0.8914 |
| 0.0001 | 0.002 | 20 | 0.9245 | 0.8947 |

表中首行对应的全局最优组合为骨干学习率 $3\times10^{-4}$、分类头学习率 $3\times10^{-3}$、训练轮次 40，验证集最佳准确率为 **0.9401**，对应测试集准确率为 **0.9062**（该表中的最高测试集准确率为 **0.9119**，出现在 $\text{blr}=3\times10^{-4},\text{hlr}=3\times10^{-3},\text{epoch}=30$ 的组合）。

从分类头学习率角度看，高分组合明显集中在较大的分类头学习率：验证集最佳准确率前 10 组中，`head_lr=0.003` 占 6 组、`head_lr=0.002` 占 4 组、`head_lr=0.001` 为 0 组，说明在该任务下提升分类头学习率更有利于快速适配 37 类决策边界；训练轮次方面，前 10 组中 40 轮与 30 轮各占 5 组，20 轮为 0 组，且表中最低值 **0.9245** 出现在 20 轮，表明仅训练 20 轮时性能上限更易受限。

从整体来看，在保证训练轮次充足的前提下，实验搜索网格内更优的策略是 **高分类头学习率+适当骨干学习率**；验证集最优准确率由基线的 **0.9271** 提升至 **0.9401**，提升约 **1.30** 个百分点。

#### 1.4 预训练消融实验

本实验在控制其他超参数设置一致的情况下，对比 **预训练初始化** 与 **随机初始化** 的训练性能，并将随机初始化训练从基线对比的20轮延长至80轮与120轮，进一步检验仅延长训练轮数能否缩小与预训练初始化的差距。

下表为基线模型与不同训练轮数的随机初始化(20/80/120代)的最佳验证集准确率、测试集准确率以及最佳验证集准确率对应的训练轮次。

| 初始化方式 | 训练轮数 | 最佳验证准确率 | 测试集准确率 | 最佳 epoch |
|------------|----------|----------------|--------------|------------|
| 预训练初始化 | 20 | **0.9271** | **0.8914** | 18 |
| 随机初始化 | 20 | 0.0911 | 0.1050 | 16 |
| 随机初始化 | 80 | 0.2526 | 0.1671 | 64 |
| 随机初始化 | 120 | 0.2865 | 0.2048 | 113 |

由表可知，随机初始化下，即使将训练延长至 **120** epoch，验证准确率最高也仅为 **0.2865**，与 **20** epoch 预训练模型（**0.9271**）仍相差约 **0.64**。这一现象意味着，随机初始化的性能不佳并不是训练轮次不足造成的，进而说明了预训练初始化相较于随机初始化的优势。

下图为消融实验四个实验组的训练集与验证集loss/accuracy曲线。

![](task1_classification/ReportFig/1_4.png)

由图可见，预训练初始化在前期即可实现快速收敛，验证损失迅速降至低位（约 **0.33**）且验证准确率稳定在 **0.91~0.93** 区间；随机初始化即便延长至 80/120 epoch，验证损失仍处于高位（约 **2.65~2.86**），准确率上限也仅约 **0.29**。这表明，对于使用随机初始化的实验组，即使继续增加训练轮数，带来的额外收益非常有限。这一解读印证了之前读表得到的结论：性能差距的关键不在训练时长，而在于预训练提供的高质量初始表征。

据此可以认为，在这一任务下，**预训练初始化** 为细粒度宠物分类提供了可迁移的通用视觉表征，显著降低了在有限样本上学习 37 类决策边界的难度；相比之下，随机初始化则更依赖大量数据与针对性架构设计，在任务1的应用场景下性能显著不及预训练初始化，且这一劣势并非由于训练轮次不足造成。

#### 1.5 引入注意力机制

实验中主要采用了两种方式在基线模型基础上引入注意力机制：一是在卷积神经网络基础上添加 SE、CBAM 注意力模块，二是采用轻量级vision Transformer架构，如ViT-Tiny，Swin-T。

##### 1.5.1 卷积骨干上的 SE、CBAM

在进入结果对比前，先简要说明两种注意力模块的基本机制。**SE（Squeeze-and-Excitation）** 先对每个通道做全局平均池化得到通道描述，再通过两层全连接与 sigmoid 生成通道权重，对原特征做按通道进行重标定，其核心是显式建模通道间的相互依赖，强化有判别力的通道并抑制冗余通道；**CBAM（Convolutional Block Attention Module）** 则在 SE 的通道注意力基础上,进一步加入空间注意力:先做通道注意力，再基于通道聚合后的特征图学习空间权重，从而同时回答‘关注哪些语义通道’与‘关注图像中的哪些位置’，以更细粒度地突出目标区域并抑制背景干扰。

在本节的实验中，标准 SE、CBAM 配置会在前述基线模型的基础上，在四个 layer 的每个 BasicBlock 后插入注意力模块；不同于此的是，`SE(high)` 、 `CBAM(high)` 仅在高层语义阶段(layer3 与 layer4)插入注意力模块，减少低层注意力干预，保留高层特征重标定。为与基线模型公平对比，并尽可能排除 20 轮训练不足带来的影响，四种注意力配置(包括基线)均额外进行了一次 40 轮训练。下面分开汇总并对比。

**1.5.1.表1. 20 epoch 对比**

| 模型配置 | 最佳验证准确率 | 测试集准确率 | 最佳 epoch |
|----------|----------------|--------------|------------|
| 基线模型 | **0.9271** | **0.8914** | 18 |
| ResNet-18 + SE | 0.8776 | 0.8607 | 16 |
| ResNet-18 + SE（high） | 0.8958 | 0.8755 | 19 |
| ResNet-18 + CBAM | 0.7969 | 0.7864 | 18 |
| ResNet-18 + CBAM（high） | 0.8828 | 0.8594 | 16 |

**1.5.1.表2. 40 epoch 对比**

| 模型配置 | 最佳验证准确率 | 测试集准确率 | 最佳 epoch |
|----------|----------------|--------------|------------|
| 基线模型（epoch=40） | **0.9297** | **0.9004** | 33 |
| ResNet-18 + SE | 0.8984 | 0.8804 | 22 |
| ResNet-18 + SE（high） | 0.9141 | 0.8907 | 31 |
| ResNet-18 + CBAM | 0.8828 | 0.8664 | 40 |
| ResNet-18 + CBAM（high） | 0.9219 | 0.8910 | 38 |

从两组对比表出发，可以发现一个一致现象：无论是 SE、CBAM 还是其 high 变体，40 epoch 的结果整体都优于 20 epoch。这说明将注意力模块实验补充到更长训练轮次是有必要且有效的，能够尽可能排除训练尚未充分展开带来的干扰，使后续比较更具解释力。

在此基础上进一步比较同类模块可见，SE(high) 与 CBAM(high) 在各项指标上均稳定优于各自的全层插入版本，表明在该任务中把注意力集中在高层语义阶段更合理：高层特征与类别判别更直接相关，而在低层过早施加注意力更容易扰动通用纹理与边缘表示。

与此同时，四种注意力配置在 40 epoch 下虽然较 20 epoch 均有改善，但所有评价指标均不及对应的基线模型，说明当前任务里引入注意力并不自动带来增益；更可能的情况是，额外的重标定机制提升了表示复杂度，却未稳定转化为更强泛化，且全层注意力对基础特征的干预偏强，收益被部分抵消。

下图为训练轮次20代的基线对照组与四个添加注意力模块实验组的训练集与验证集loss/accuracy曲线。

![](task1_classification/ReportFig/1_5_1_ep20.png)

下图为训练轮次40代的基线对照组与四个添加注意力模块实验组的训练集与验证集loss/accuracy曲线。

![](task1_classification/ReportFig/1_5_1_ep40.png)

从 20 与 40 轮训练两组曲线的联合对比可见，延长训练能使各模型进一步收敛，但模型间相对关系基本稳定：baseline 始终保持最低验证损失与最高验证准确率，且在前几个 epoch 即快速进入平台区，表现出更快、更稳定的收敛；SE(high)、CBAM(high)整体优于各自全层插入注意力模块(SE、CBAM)版本，曲线表现为下降更平滑、后期震荡更小，说明将注意力集中于高层语义阶段比全层插入更有效；其中全层 CBAM 收敛最慢、前中期提升最滞后且最终平台值最低，体现出明显的优化困难与性能上限受限。以上实验现象进一步印证了表格结果解读的结论。

综合而言，本节结果支持两点结论：其一，相比全层插入注意力模块，在该任务中将注意力集中在高层语义阶段更合理；其二，四种加入注意力模块的架构均未形成对基线的稳定超越，提升的复杂度并未有效转化为更强的泛化性能。

##### 1.5.2 轻量级 Transformer（ViT-Tiny、Swin-T）及训练配方对齐的 ResNet 对照

在进入结果对比前，先简要说明三类模型的表征机制。作为对照基线，**ResNet-18** 依赖卷积的局部感受野与层级特征提取，具有较强的局部归纳偏置；**ViT-Tiny** 先把输入图像按固定大小切成不重叠 patch，每个 patch 展平后线性映射为 token，再与位置编码相加后送入 Transformer Encoder：在每一层中，任意 token 都可与其余所有 token 计算注意力，因此能直接建模跨区域、长距离的依赖关系。其代价是注意力计算复杂度随 token 数近似二次增长，输入分辨率提高时计算与显存开销上升明显；**Swin-T** 则将自注意力限制在局部窗口内（W-MSA），并在相邻层采用移位窗口（SW-MSA）让原本不在同一窗口的 token 建立连接，同时通过 patch merging 逐阶段降低分辨率、提升通道数，形成类似 CNN 的金字塔特征：这样既把注意力复杂度从全局二次关系降到更可控的局部规模，又保留了跨窗口信息流动和多尺度语义表达能力，更适配高分辨率视觉任务。

由于三者的归纳偏置与信息交互方式存在本质差异，需在统一训练配方下进行对照，才能相对公平地比较架构本身的有效性。因此，除 ViT-Tiny、Swin-T 外，另训练一条采用了Transformers训练配方的 **ResNet-18** 基线：三者均采用 **AdamW**（骨干学习率 $5\times10^{-5}$、分类头学习率 $5\times10^{-4}$）、**权重衰减 0.05**、**batch size 32**、**30 epoch**、**warmup 余弦**调度（ warmup 轮次为 5，`warmup_start_factor` 为 **0.001**），并在 ImageNet 预训练权重上微调。下表据此与 ViT-Tiny、Swin-T 进行公平对比。

| 模型 | 最佳验证准确率 | 测试集准确率 | 最佳 epoch |
|------|----------------|--------------|------------|
| ResNet-18（对齐Transformer配置） | **0.9349** | 0.9056 | 24 |
| ViT-Tiny | **0.9427** | 0.9087 | 25 |
| Swin-T | **0.9609** | 0.9351 | 13 |

由表可知，在对齐优化器与调度后，ViT-Tiny 相比对照 ResNet-18 已表现出稳定优势：验证集最佳准确率由 **0.9349** 提升至 **0.9427**（+**0.78** 个百分点），测试集准确率由 **0.9056** 提升至 **0.9087**（+**0.31** 个百分点）。也即，ViT-Tiny 在验证与测试两侧均优于基线，但提升幅度并不完全一致；相比之下，**Swin-T** 的验证/测试准确率分别达到 **0.9609/0.9351**，相对 ViT-Tiny 进一步提升约 **1.82/2.64** 个百分点，显示出更明显的领先优势。这一现象表明，`层次化窗口注意力`相较于 patch 式全局注意力与卷积局部归纳偏置，更有利于捕捉多尺度空间上下文与细粒度品种差异。

下图为对齐Transformers配置的基线对照组与ViT-Tiny架构、Swin-T架构实验组的训练集与验证集loss/accuracy曲线。

![](task1_classification/ReportFig/1_5_2.png)

从三组模型的验证曲线对比可以看到，在统一训练配方下三者均快速收敛，但收敛速度与平台上限差异明显。Swin-T 实验组在前 2-4 个 epoch 内即完成主要性能爬升：val_loss 快速降至最低区间并在全程保持优势，val_acc 也最早进入高平台（约 0.95 左右）并长期领先。ViT-Tiny 与对照 ResNet-18 均在约第 8-12 个 epoch 后进入稳定阶段，但二者之间亦有细微差异：ViT-Tiny 的 val_loss 整体低于对照组、val_acc 平台也更高（约 0.94 vs 0.93），对应其表格中验证与测试指标均小幅领先。

从优化轨迹看，Swin-T 不仅最终指标更优，而且其优势在训练早期就已建立并持续保持，意味着其层次化窗口注意力在该任务上具有更高的优化效率：在相同 epoch 预算内更快获得有效表征，并在后续阶段维持更低损失和更高精度。ViT-Tiny 虽然同样具备全局建模能力，但其表现介于 Swin-T 与卷积基线之间：相较对照 ResNet-18 已形成稳定提升，却未达到 Swin-T 的收敛速度与收敛上限。

结合表格解读与曲线解读可以认为，在当前实验配置下三者形成清晰分层：Swin-T 在收敛速度与收敛上限两个维度上均显著领先；ViT-Tiny 在验证与测试两侧均优于对齐 Transformers 配置的 ResNet-18 基线，但领先幅度有限。整体上，Swin-T 仍是本节乃至整个任务1中性能最优的训练方案。

#### 1.6 小结

任务1的实验结论可以概括为四点。

第一，在统一基线设置下进行 27 组网格搜索后可以得知，较优的学习率组合(**适当骨干学习率+高分类头学习率**)配合**较多的训练轮次**可带来明确增益：以验证集性能最优组合（$\text{blr}=3\times10^{-4},\text{hlr}=3\times10^{-3},40$ epoch）为例，最佳验证准确率由基线 **0.9271** 提升至 **0.9401**（+**1.30** 个百分点），对应测试集准确率由 **0.8914** 提升至 **0.9062**（+**1.48** 个百分点）；

第二，预训练初始化的优势非常显著：随机初始化后，即使将训练轮次延长到 120 轮，最佳验证准确率仍仅为 **0.2865**，与预训练初始化后训练 20 轮得到的最佳验证准确率 **0.9271** 存在大幅差距，这一消融实验结果表明，仅通过延长随机初始化模型的训练轮次，难以弥补其与预训练初始化之间的性能差距，这意味着在本实验设置下，预训练带来的初始表征质量是获得高性能的关键因素之一；

第三，在卷积骨干中引入 SE、CBAM 注意力模块时，仅在高层引入注意力模块的变体(SE(high)、CBAM(high))最佳验证准确率整体高于在所有层引入注意力模块(SE、CBAM)，表明将注意力集中于高层语义阶段更有效，但无论训练轮次设置为 20 抑或是 40 ，引入注意力的模型性能均未稳定超过对应基线，说明引入注意力模块提升的复杂度，在当前实验设置下并未有效转化为更强的泛化性能；

第四，在与 Transformer 对齐的训练配方下，若以**最佳验证集准确率**衡量，ResNet-18、ViT-Tiny、Swin-T 分别为 **0.9349**、**0.9427**、**0.9609**；若以**测试集准确率**衡量，三者分别为 **0.9056**、**0.9087**、**0.9351**。其中 ViT-Tiny 在验证与测试两项指标上均优于对照 ResNet-18，但两者都明显低于 Swin-T；这表明**层次化窗口注意力**在捕捉多尺度空间上下文与细粒度品种差异方面，相较于 patch 式全局注意力与卷积局部归纳偏置更具优势；patch 式全局注意力虽亦优于卷积局部归纳偏置，但其在任务1场景下的性能提升不及层次化窗口注意力。


### 2. **任务2**：场景目标检测与视频多目标跟踪

本任务以 VisDrone 无人机航拍场景为对象，围绕检测、跟踪与计数三步流程展开：先在检测数据集上微调 YOLOv8 得到专属检测器，再将其用于测试视频逐帧推理与多目标跟踪，最后结合虚拟越线规则统计跨线目标总数。与任务1、任务3侧重分类/分割不同，任务2更强调模型在复杂交通场景中的时序稳定性，即不仅要给出每一帧的类别与边界框，还需要在跨帧过程中尽量维持同一目标的 Tracking ID 一致。

在检测性能上，本文以验证集 precision、recall、mAP50 与 mAP50-95 作为主要指标；在视频分析上，重点关注遮挡与密集交汇场景下的 ID 连续性，以及越线计数逻辑是否满足同一轨迹仅计算一次的约束。

#### 2.1. 数据集与公共设置说明

检测训练数据采用 VisDrone 官方检测集划分，训练与验证路径由 `configs/visdrone_local.yaml` 统一指定（`train` 与 `val` 分别指向对应图像目录，标签由同级 YOLO 格式 `labels/` 提供），类别数为 **10**（pedestrian、people、bicycle、car、van、truck、tricycle、awning-tricycle、bus、motor）。原始标注先通过脚本转换为 YOLO 检测格式，再进入统一训练流程。

三次检测实验的公共训练设置保持一致：`epochs=50`、`imgsz=640`、`batch=8`，其余优化超参数使用 Ultralytics 默认配置（含 warmup 与数据增强策略）；仅主干模型规模不同，使用不同制式的Yolov8模型。

在此之后，使用训练得到的最优检测器，对测试视频逐帧进行目标检测，并在相邻帧之间完成目标关联，从而为同一实例分配尽可能稳定的跟踪编号。可视化结果会同时展示目标框、类别和跟踪编号，同时保存逐帧轨迹记录，便于后续遮挡片段分析与误差排查。越线统计基于目标中心点相对虚拟线的位置变化进行判定，并采用“同一目标仅在首次有效穿越时计数”的策略，避免同一目标往返运动造成重复累计。

#### 2.2. 评价指标与判定准则说明

设验证集中预测框集合为 $\mathcal{D}=\{(b_i,s_i,c_i)\}$，其中 $b_i$ 为边界框，$s_i\in[0,1]$ 为置信度，$c_i$ 为类别；真实框集合为 $\mathcal{G}=\{(g_j,\tilde c_j)\}$。对类别 $c$，先按置信度从高到低对预测排序，在 IoU 阈值 $\tau$ 下做一一匹配（同一真实框最多匹配一次）：

- 匹配成功记为 $\mathrm{TP}$；
- 未匹配预测记为 $\mathrm{FP}$；
- 未被任何预测匹配的真实框记为 $\mathrm{FN}$。

据此可得
$$
\mathrm{Precision}(\tau)=\frac{\mathrm{TP}(\tau)}{\mathrm{TP}(\tau)+\mathrm{FP}(\tau)}
$$
$$
\mathrm{Recall}(\tau)=\frac{\mathrm{TP}(\tau)}{\mathrm{TP}(\tau)+\mathrm{FN}(\tau)}
$$

当扫描不同置信度阈值时，可得到该类别的 Precision-Recall 曲线。其面积定义为该类别在 IoU 阈值 $\tau$ 下的平均精度：
$$
\mathrm{AP}_c(\tau)=\int_0^1 p_c(r;\tau)\,dr,
$$
其中 $p_c(r;\tau)$ 表示经插值后的 PR 曲线。

多类别平均后得到
$$
\mathrm{mAP}(\tau)=\frac{1}{C}\sum_{c=1}^{C}\mathrm{AP}_c(\tau),
$$
其中 $C=10$ 为 VisDrone 检测类别数。于是：

- **mAP50** 对应 $\mathrm{mAP}(0.50)$；
- **mAP50-95** 对应 COCO 风格平均：
$$
\mathrm{mAP}_{50\text{-}95}=\frac{1}{10}\sum_{k=0}^{9}\mathrm{mAP}(0.50+0.05k).
$$

其中 mAP50 对定位误差更宽容，更能反映是否检测到了目标；mAP50-95 覆盖更严格的 IoU 区间，对定位质量与框回归精度要求更高，因此通常数值低于 mAP50。

#### 2.3. 检测模型训练过程与实验结果对比

在本实验中，尝试了三个不同制式的YOLOv8模型作为候选模型。在本节中给出训练过程演示以及实验结果对比。

下图为YOLOv8n/s/m模型的验证集 mAP50 与验证集 mAP50-95 随训练轮次变化的曲线示意图。

![](task2_detection_tracking/ReportFig/2_3_mAP.png)

下图为YOLOv8n/s/m模型的验证集准确率与验证集召回率随训练轮次变化的曲线示意图。

![](task2_detection_tracking/ReportFig/2_3_PR.png)

纵向对比来看，两张图中的各曲线整体呈现一致趋势：三个模型的四个不同验证集指标在前 10~15 个 epoch 提升最快，随后进入缓慢爬升并逐步趋于平台，说明训练过程稳定、后期增益有限；横向对比上，YOLOv8m 在 mAP50、mAP50-95、Precision 和 Recall 四类指标上整体保持领先，YOLOv8s 次之，YOLOv8n 相对较低，体现出模型容量增大带来的显著检测性能收益；与此同时，三组实验中 Precision 普遍高于 Recall，且差距在中后期仍存在，说明当前设置下模型对误检控制较好，但仍有一定漏检空间。综合 mAP 与 PR 曲线可得：YOLOv8m 具有最佳精度-召回折中与最优定位质量，但在最终选定其为应用于后续实验的最优检测器之前，仍需要结合日志中的验证集指标信息进行验证。

下表为三个模型的最终训练结果，展示了验证集mAP50最高的轮次对应的轮次序号、mAP50取值、mAP50-95取值、准确率与召回率。

| 模型名 | 最优轮次 | mAP50 | mAP50-95 | Precision | Recall |
|------|-----|------|------|------|------|
| yolov8n.pt | 49 | 0.2961 | 0.1663 | 0.4210 | 0.3226 |
| yolov8s.pt | 39 | 0.3792 | 0.2204 | 0.5102 | 0.3928 |
| yolov8m.pt | 38 | 0.4246 | 0.2524 | 0.5496 | 0.4267 |

下表给出三个模型在最后一轮（epoch=50）的验证集指标：

| 模型名 | mAP50 | mAP50-95 | Precision | Recall |
|------|------|------|------|------|
| yolov8n.pt | 0.2969 | 0.1661 | 0.4249 | 0.3224 |
| yolov8s.pt | 0.3813 | 0.2201 | 0.5101 | 0.3952 |
| yolov8m.pt | 0.4221 | 0.2502 | 0.5614 | 0.4240 |

结合两张图显示的各项验证集指标可以看到，随模型容量从 n/s/m 逐步增大，四项检测指标整体呈上升趋势， 模型容量最大的 YoloV8 版本在验证集上取得当前最优结果。因此，Visdrone 微调后的 `yolov8m.pt` 被选为后续测试视频逐帧推理与多目标跟踪的最佳检测器。

#### 2.4. 视频多目标跟踪与ID稳定性分析

##### 2.4.1. 检测目标密集场景下的目标丢失案例

实验中，通过人工甄别，选取以第 **624** 帧至 **627** 帧的四帧作为典型的密集场景目标丢失片段。

![](task2_detection_tracking/ReportFig/2_4_1.png)

以画面中心左侧 `id:1` 的 `truck` 为参照，可观察其上下两侧 `motor` 检测框在 623-627 五帧内都出现了同类“短时缺失”变化：在 `truck` 上方的一辆摩托仅在第 **624** 与 **627** 帧出现，第**625** 与 **626** 帧均无检测框；在 `truck` 下方的两辆摩托中，更靠下的一辆在第 **625** 帧消失，其余帧均被检测到。两处都属于同一模式——目标在局部时段短暂掉检，随后恢复检测——共同指向密集场景下检测稳定性下降导致的短时目标丢失。

判定结果上，该片段主要表现为**目标丢失**，而不是明确的 ID 互换。其成因可以从检测与关联两个环节理解。首先，在密集交汇与局部遮挡条件下，目标可见区域明显缩小，边缘与纹理信息不完整，小尺度目标的检测置信度更容易波动并跌破阈值，导致个别帧检测框缺失。其次，跨帧关联需要依据运动预测和空间邻近完成匹配，当某一帧缺少可靠候选框时，轨迹只能短时外推；若后续仍无法与有效检测结果匹配，轨迹就会进入临时失联状态，表现为有效个体数量下降。待目标重新露出并恢复可检测后，匹配关系再次建立，个体数随即回升。该片段因此体现了密集场景下遮挡引发的检测不连续，并进一步传导为跟踪层面的短时目标丢失。

##### 2.4.2. 检测目标密集场景下的ID分裂案例

该案例来自人工逐帧观察：在第 **982-985** 帧附近，画面左侧道路上方，同一车辆位置(在 982 帧中对应检测框 `id:1236` )出现id标注不一致的跟踪框，属于明显的ID相关异常窗口；但仅凭画面尚无法直接区分其是ID跳变，还是重复轨迹/ID分裂。对应关键帧按时间顺序如下：

![](task2_detection_tracking/ReportFig/2_4_2.png)

从时间连续性上看，第 982 帧尚未明显异常，`id:1236` 框正常显示；到第 983 与第 984 帧，第 982 帧的几乎同一位置出现了不同的框:`id:1816` 的框在可视化中替代了 `id:1236` 的位置；第 985 帧该区域的检测框又恢复为`id:1236`。基于图像本身，只能先将该片段判为ID相关异常窗口，其性质既可能是 ID 跳变，也可能是 ID 分裂/重复轨迹，尚不能仅凭画面直接定性。

因此在人工观察之后，需要回到 `track_log.csv` 做定量求证。进一步补充抽取 982/983/984/985 帧中相关记录（仅保留 `id, cx, cy, cls`）如下：

| 帧号 | id | cx | cy | cls |
|------|------|------|------|------|
| 982 | 1236 | 141.17 | 190.45 | 4 |
| 983 | 1236 | 141.90 | 190.29 | 4 |
| 983 | 1816 | 141.61 | 190.37 | 3 |
| 984 | 1236 | 142.75 | 190.26 | 4 |
| 984 | 1816 | 142.26 | 190.36 | 3 |
| 985 | 1236 | 143.58 | 190.24 | 4 |

从表中可以看到：`id=1236` 在 982→985 帧位置连续且平滑移动，而 `id=1816` 仅在 983/984 两帧短时出现，且其中心点与 `id=1236` 在同帧内几乎重合。这一结果更支持重复轨迹/ID分裂，而非ID跳变的判断。

检测对象密集场景下出现的重复轨迹/ID分裂现象成因，可从密集场景下的**检测抖动 + 关联不确定性**共同解释。结合上文 982 到 985 帧的时序现象与日志表格可见，`id=1236` 的中心点在该窗口内保持连续平滑，而 `id=1816` 仅在 983/984 两帧短时出现且与 `id=1236` 高度重合，这更符合**同一目标被临时复制出第二条轨迹**的模式。其机理是：在密集交汇区域，目标间遮挡、相似外观与边界框轻微漂移会降低跨帧匹配的稳定性；当候选匹配关系落在 IoU/距离门控阈值附近时，关联器可能未及时终止旧轨迹，同时又激活新轨迹，于是产生并行 ID。类别分支在相近类别（如 `car/van`）上的短时波动还会进一步增加这种误关联的可见性。该案例因此反映了：在密集场景中，即使几何运动连续，跟踪ID也可能因短时关联不稳定而发生重复轨迹/ID分裂。

##### 2.4.3. 检测目标密集场景下的ID跳变案例

与 2.4.3 相同，本节先由关键帧做时序观察，再回到 `track_log.csv` 进行性质确认。本小节选取第 **840-842** 帧作为典型片段。

![](task2_detection_tracking/ReportFig/2_4_3.png)

从画面连续性看，三帧中的目标位于相近空间位置并沿道路方向平滑移动，但标签 ID 在中间帧发生变化：第 840 帧为 `id:417`，第 841 帧变为 `id:1528`，第 842 帧又回到 `id:417`。类似于 2.4.2 中的情况，无法通过视频断定出现的错误是ID跳变，还是重复轨迹/ID分裂。

因此，进一步抽取 `track_log.csv` 中 840/841/842 帧相关记录（仅保留 `id, cx, cy, cls`）如下：

| 帧号 | id | cx | cy | cls |
|------|------|------|------|------|
| 840 | 417 | 55.19 | 165.04 | 0 |
| 841 | 1528 | 57.37 | 166.25 | 0 |
| 842 | 417 | 55.03 | 164.67 | 0 |

从表中可见，三帧中心点坐标连续且空间距离很近，类别 `cls` 保持一致，但不同于上一节，`id:417` 并没有在ID异常过程中一直保持存在：轨迹 ID 出现了 `417 -> 1528 -> 417` 的短时切换，第 841 帧没有出现 `id:417` 检测框的记录。这表明每一帧仅出现一个对应 ID，不符合同帧中不同 ID 框在紧邻位置重叠的 ID 分裂特征，更符合短时关联重置导致的 ID 跳变。

该片段可用**关联短时失稳—轨迹临时重建—旧轨迹回连**的机制链解释。跟踪器在相邻帧并不是直接沿用上一帧 ID，而是先依据运动连续性（如上一时刻速度外推后的预测位置）与空间相似性（检测框位置/重叠关系）计算匹配代价，再在门限约束下完成关联。密集场景中一旦出现轻微遮挡、邻近目标干扰或检测框抖动，这个匹配代价就可能在阈值附近来回波动。

结合本例可更具体地理解：第 840 帧 `id:417` 关联正常；到第 841 帧，目标仍在原邻域内，但匹配分数短时跌破门限，旧轨迹未被成功匹配，系统转而把当前检测结果初始化为新轨迹 `id:1528`；第 842 帧目标外观与位置再次稳定后，旧轨迹 `id:417` 又重新获得匹配，从而出现 `417 -> 1528 -> 417` 的短窗切换。由于 `cx, cy` 连续且 `cls` 不变，这种模式更像同一目标在关联层发生瞬时重置，而不是两个不同目标的真实交接。

因此，本案例揭示的核心不是运动不连续，而是关联决策处于门限边界时的鲁棒性不足：几何轨迹连续并不必然保证 ID 连续，在密集交汇区域仍可能出现可见的短时 ID 跳变。

##### 2.4.4. 特殊**同ID类别跳变**案例

在作业要求的**密集交汇场景下发生ID跳变或目标丢失**基础上，实验中还在输出视频里额外观察到一类特殊错误，也即，检测目标的ID保持连贯，但分类标识跳变的案例。与 2.4.3 的分析流程一致，本节同样先由画面定位异常，再结合时序连续性与日志信息做性质判定。本小节选取第 **274-277** 帧作为人工甄别出的典型片段。该片段整体目标周围干扰并不严重，但仍出现了短时丢失结合同ID类别跳变的复合不稳定现象。

![](task2_detection_tracking/ReportFig/2_4_4.png)

关注右侧道路上方的摩托车目标并进行逐帧解读。第 **274** 帧该目标未被分配检测框；第 **275**、**276** 帧被分配为 `id:500` 的 `motor` 框，这表明该目标经历了短时漏检；第 **277** 帧仍保持 `id:500`，但类别由 `motor` 变为 `bicycle`。有趣的是，道路左侧 `id:366` 检测目标与 `id:500` 发生了截然相反的转变：其在**276** 帧被标记为 `bicycle` 类，却在第 **277** 帧变为 `motor` ，其id亦保持不变。这表明目标被检出并保持同一轨迹ID，但分类结果在相邻帧发生漂移。

从判定结果看，这组帧中跟踪器对两个主体总体上是**ID维持成功**（`id:500` 与 `id:366` 均未在该窗口发生ID重置），但出现了“同ID类别跳变”。这与多目标跟踪的工作分工一致：关联模块主要依据时序运动连续性与空间邻近关系（如相邻帧IoU、运动预测门限）来决定**是不是同一目标**，并不把类别一致性作为硬约束；因此只要轨迹几何连续，`track_id` 可以保持稳定。另一方面，检测分类分支在小目标、视角变化、局部遮挡和背景干扰下更易在相近类别间振荡，`motor` 与 `bicycle` 尤其容易互相混淆。更关键的是，本片段中 `id:366` 与 `id:500` 在同一时间窗口发生了方向相反但形态一致的类别漂移，这更像是分类头在相邻细粒度类别上的系统性判别边界不稳定，而非单个目标的偶发误判。该案例因此体现了：在稀疏场景中，算法可以维持ID连续，但类别语义仍可能短时失稳。

#### 2.5. 越线计数结果与关键帧说明

本节越线计数实验基于 `output.mp4` 的画面分辨率 **640×360** 进行，虚拟线采用像素坐标 **(120, 100) → (560, 100)**，为画面靠上方一条横向线段。后续所有越线候选筛选、成功/失败案例分析均以该画面尺寸与该线参数为判定基准。

实验`infer_track.py`的越线计数采用以下判定逻辑：先按 `track_id` 与帧序对每条轨迹排序，逐帧计算目标中心点相对虚拟线的符号（位于直线两侧分别记为正/负，在线上记为 0）；当同一轨迹出现非零符号翻转（由正到负或由负到正）时，判定目标发生一次穿越，并将该时刻计为一次有效越线事件（实时统计时左上角记录的越线数加 1，离线记录时记录为候选中间帧）。为避免同一目标反复往返造成重复计数或重复候选，两处实现都仅保留每个 `track_id` 的首次有效穿越事件。

在该判定下，几类典型越线计数错误可对应到如下数据形态：

- **漏计（False Negative）**：真实目标已越线，但在越线附近帧中 `track_log` 缺失该目标的连续记录（无稳定 `track_id`，或 `cx,cy` 无法形成符号翻转），表现为视觉上穿线但日志中无对应跨线事件；
- **误计（False Positive）**：目标未真实越线，但受检测框抖动影响，检测框中心坐标在虚拟线两侧来回摆动，造成符号短时翻转并触发计数；
- **ID切换引发计数偏差**：目标在越线附近发生 `old_id -> new_id`，使单目标轨迹被拆分，可能出现“应计而未计”或“重复计数”；
- **遮挡场景关联失败**：多个相邻目标在交汇区发生短时重叠，日志中出现轨迹中断或重新关联，导致计数事件主体映射不稳定。

根据`Output.mp4`左上角的计数框，视频中发生了一共25次越线。下图为输出视频的最后一帧，可以通过左上角的黄色计数来验证。

![](task2_detection_tracking/ReportFig/2_5.png)

经人工逐帧筛查，本输出视频中没有出现前述提到的后三类错误，肉眼观察到的漏记有4到5次。为更完整展示越线计数逻辑，本节给出一个典型成功越线计数案例与一个典型的越线漏计失败案例。

##### 2.5.1. 成功越线计数案例

报告选取周围无其他个体干扰视野的清晰越线帧 **185** 作为可视化案例。由 `frame_000184` 到 `frame_000185` 可观察到id为141的汽车目标检测框中心点由线的一侧移动到另一侧，触发一次有效穿越计数。

![](task2_detection_tracking/ReportFig/2_5_1.png)

##### 2.5.2. 失败越线计数案例(漏计)

报告选取的失败越线计数(漏记)典型片段发生在第18至21秒期间。在这一漏记片段中，可见画面左上角人行道有一名白衣行人在从左图到右图的运动过程中实际发生了跨线运动，但该目标在相关时段未被稳定检测到（未分配有效检测框与跟踪ID），因此未触发越线计数，左上角越线计数显示值在片段过程中保持不变(18)佐证了这一点。这说明当前计数结果高度依赖检测与跟踪的连续性：当检测框缺失时，即使真实穿线发生，系统也会出现漏计。

![](task2_detection_tracking/ReportFig/2_5_2.png)

#### 2.6 小结

任务2的实验结论可以概括为三点。

第一，在统一训练配置（`epochs=50`、`imgsz=640`、`batch=8`）下，模型规模增大带来稳定的检测性能提升：验证集 mAP50 从 YOLOv8n 的 **0.2961** 提升到 YOLOv8m 的 **0.4246**，测试集 mAP50 从 **0.2969** 提升到 **0.4221**，说明在当前 VisDrone 场景中更强 backbone 具有明确收益；

第二，从跟踪可视化与日志排查看，密集场景中的误差类型具有不同成因：短时目标丢失主要由遮挡、尺寸过小或检测置信度波动导致的漏检引起；ID分裂通常发生在轨迹短时中断后重现，被关联模块分配新 `track_id`；ID跳变更多出现在近邻目标交汇或相互遮挡时，关联代价在多目标间误匹配；同ID类别跳变则主要来自检测分类头在相近类别（如 `motor` / `bicycle`）上的判别边界不稳定。综合来看，当前方案在复杂遮挡与近邻交汇场景下的时序一致性仍受检测稳定性与关联鲁棒性共同约束；

第三，越线计数模块在规则定义上是可解释且可复现的：基于目标中心点跨越有向线段的侧别翻转判定，且同一 `track_id` 仅计首次有效穿越；在本实验设置（`line=(120,100)->(560,100)`，分辨率 `640×360`）下，最终统计越线次数为 **25**，并可由输出视频末帧计数框验证。结合成功/失败案例可见，计数误差主要来源于检测/跟踪链路连续性不足而非计数规则本身：检测稳定时可准确计数，越线关键帧若缺少稳定检测框与轨迹ID则会漏计，因此后续优化应优先提升密集场景下的检测召回与跟踪连续性。

### 3. **任务3**：从零搭建与损失函数工程：图像分割模型的像素级训练

本任务聚焦于三分类语义分割：在不使用预训练权重的前提下，从随机初始化开始训练 U-Net，对 Oxford-IIIT Pet 数据集中的每个像素预测其类别（前景宠物、宠物边界、背景）。实验目标不是比较不同的网络结构，而是比较在相同的U-Net架构下，不同损失函数设计在训练过程中的优化行为与分割质量差异。

整体实验包含三种损失配置：仅 Cross-Entropy（CE）、仅 Dice、以及 CE 与 Dice 的加权组合。评价指标采用验证集 mIoU，以衡量模型在类别重叠意义下的像素级分割效果。

#### 3.1. 实验设置说明

##### 3.1.1. 数据集与公共设置说明

数据集采用 Oxford-IIIT Pet 官方图像与 trimap 标注。实验将官方 `trainval`（3680 张）按 `8:2` 随机划分为训练集（2944 张）与验证集（736 张），并固定随机种子为 42 以保证划分一致与结果可复现。训练集仅使用随机水平翻转（概率 0.5）作为数据增强，验证集不做数据增强；二者均统一 resize 到 256×256，图像像素均缩放到 `[0,1]` 区间。

在上述数据与预处理设置下，任务3共进行了两轮对比实验，且每轮均包含三种损失函数（CE、Dice、CE+Dice）：

- 第一轮：`epochs=30`；
- 第二轮：`epochs=50`，并启用早停 `patience=10`。

两轮实验其余公共训练配置保持一致：batch size 为 8，优化器为 AdamW（`lr=1e-3`，`weight_decay=1e-4`），默认使用 cosine 学习率调度。每次训练结束后均加载最佳验证模型，并在官方 `test` 划分上进行一次最终评估，将 `test_loss` 与 `test_mIoU` 写入结果文件。

分割类别数记为 $C=3$。对任一 batch，记样本数为 $N$，空间分辨率为 $H\times W$，模型输出 logits 为 $z\in\mathbb{R}^{N\times C\times H\times W}$，像素级标签为 $y\in\{0,1,2\}^{N\times H\times W}$。

##### 3.1.2. 网络结构设置说明

本实验采用对称的 U-Net 编码器-解码器结构：输入为三通道图像，输出为三类像素分类结果。网络深度设置为 **4 次下采样 + 4 次上采样**，并在每个解码阶段与对应编码阶段执行同尺度特征融合（skip connection）。

设基础通道数为 64，则编码端通道宽度按层级递增为
$$
64 \rightarrow 128 \rightarrow 256 \rightarrow 512 \rightarrow 1024,
$$
在输入尺寸为 $256\times256$ 时，空间分辨率对应递减为
$$
256 \rightarrow 128 \rightarrow 64 \rightarrow 32 \rightarrow 16.
$$
解码端按对称方式逐级恢复分辨率，通道宽度依次回退为
$$
512 \rightarrow 256 \rightarrow 128 \rightarrow 64,
$$
最终通过逐像素线性映射得到 $C=3$ 类 logits。

具体来说，解码阶段每上采样到一个尺度，都会与编码阶段对应尺度的特征进行拼接融合，再经过卷积变换得到新的解码特征。这样可以把高层语义信息与浅层细节信息结合起来，在保证目标区域语义一致性的同时，更好地恢复边界和细粒度轮廓。

卷积子块采用**两次卷积非线性变换**的标准形式，在各尺度上完成局部纹理提取与语义表征增强；skip connection 则提供高分辨率定位信息，是提升边界类别分割质量的关键结构。

#### 3.2. 实验配置原理说明

##### 3.2.1. U-Net 网络结构设置说明

本任务采用标准编码器-解码器式 U-Net。设输入图像为 $x\in\mathbb{R}^{3\times H\times W}$，其中H、W为图片的高与宽。编码端通过多层卷积块与下采样逐步提取高语义特征，空间分辨率逐层减小、通道数逐层增大；解码端通过上采样逐步恢复空间分辨率，并与对应尺度的编码端特征进行 skip connection 拼接，以同时保留高层语义与低层细节。

编码端的下采样与多层卷积属于典型 CNN 分层特征提取流程（局部感受野逐层扩大、语义逐层增强），为避免与常规卷积网络重复，本文不再展开。下面主要解释 U-Net 如何从低分辨率高语义特征恢复到像素级预测分辨率。

若第 $l$ 层编码特征记为 $E_l$、解码特征记为 $D_l$，则可抽象写为
$$
D_l = \phi_l\big(\operatorname{Concat}(\operatorname{Up}(D_{l+1}), E_l)\big),
$$
其中 $\operatorname{Up}(\cdot)$ 表示上采样，$\phi_l(\cdot)$ 表示该层卷积变换。

这里的 skip connection 采用“通道维拼接（Concat）”而不是直接相加：将编码端同尺度特征 $E_l$ 与上采样后的解码特征共同输入后续卷积块。这样做的作用有两点：一是把浅层的边缘、纹理和位置信息直接传递到解码端，缓解上采样过程中的细节丢失；二是为高层语义特征提供精细定位参照，从而提高目标轮廓与边界区域的分割质量。

从解码特征到最终输出的过程可进一步写为：先得到最高分辨率解码特征 $D_1$，再通过分类头（$1\times1$ 卷积）在每个像素位置做通道映射，得到
$$
z = \psi(D_1),\quad z\in\mathbb{R}^{N\times C\times H\times W},\ C=3,
$$
其中 $\psi(\cdot)$ 表示逐像素线性分类映射。随后在类别维做 softmax：
$$
p_{n,c,h,w}=\frac{\exp(z_{n,c,h,w})}{\sum_{k=1}^{C}\exp(z_{n,k,h,w})},
$$
从而得到每个像素属于三类（前景、边界、背景）的概率分布。该结构对宠物边界与细粒度轮廓恢复较为友好，适合本任务的三分类像素级预测。

##### 3.2.2. mIoU 计算原理说明

设模型预测类别图为 $\hat{y}\in\{0,1,2\}^{N\times H\times W}$，真实标签为 $y\in\{0,1,2\}^{N\times H\times W}$。对每个类别 $c\in\{0,1,2\}$，定义
$$
\mathrm{IoU}_c=\frac{|\{\hat{y}=c\}\cap\{y=c\}|}{|\{\hat{y}=c\}\cup\{y=c\}|}.
$$
当某一类在当前 batch 中并未出现（即分母为 0）时，该类 IoU 不参与平均。最终 mIoU 定义为有效类别 IoU 的算术平均：
$$
\mathrm{mIoU}=\frac{1}{|\mathcal{C}_{\text{valid}}|}\sum_{c\in\mathcal{C}_{\text{valid}}}\mathrm{IoU}_c,
$$
其中 $\mathcal{C}_{\text{valid}}=\{c\mid |\{\hat{y}=c\}\cup\{y=c\}|>0\}$。

该定义能够在多类别分割中同时衡量前景、边界与背景的重叠质量，并避免“空类别”对均值造成不必要干扰。

##### 3.2.3. cosine 学习率调度原理说明

训练采用余弦退火学习率调度（Cosine Annealing）。设初始学习率为 $\eta_{\max}$、最小学习率为 $\eta_{\min}$，训练进度记为 $t$，一个退火周期长度为 $T_{\max}$，则学习率随训练进度按
$$
\eta_t = \eta_{\min}+\frac{1}{2}(\eta_{\max}-\eta_{\min})\left(1+\cos\frac{\pi t}{T_{\max}}\right)
$$
平滑下降。

在本实验中，默认 $\eta_{\max}=10^{-3}$，$\eta_{\min}=10^{-6}$；若未额外指定周期长度，则取 $T_{\max}=\text{epochs}$。调度器按 epoch 更新，因此学习率会从初始值逐步退火到接近最小值，避免后期更新步长过大带来的震荡。

##### 3.2.4. 损失函数计算原理说明

为便于统一描述，先定义像素级 softmax 概率：
$$
p_{n,c,h,w}=\frac{\exp(z_{n,c,h,w})}{\sum_{k=1}^{C}\exp(z_{n,k,h,w})},
$$
其中 $n$ 表示样本索引，$c$ 表示类别索引，$(h,w)$ 表示像素位置。

**(1) Cross-Entropy（CE）损失**
$$
\mathcal{L}_{\mathrm{CE}}=-\frac{1}{NHW}\sum_{n=1}^{N}\sum_{h=1}^{H}\sum_{w=1}^{W}\log p_{n,y_{n,h,w},h,w}.
$$
该损失从像素分类角度优化，鼓励每个像素在真实类别上的预测概率尽可能高。

**(2) Dice 损失**

先将标签 one-hot 化为 $g_{n,c,h,w}\in\{0,1\}$，并在类别维上计算 Dice 系数：
$$
\mathrm{Dice}_c=\frac{2\sum_{n,h,w}p_{n,c,h,w}g_{n,c,h,w}+\epsilon}{\sum_{n,h,w}p_{n,c,h,w}+\sum_{n,h,w}g_{n,c,h,w}+\epsilon},
$$
其中 $\epsilon>0$ 为平滑项（防止分母为零）。多类别 Dice 损失记为
$$
\mathcal{L}_{\mathrm{Dice}}=1-\frac{1}{C}\sum_{c=1}^{C}\mathrm{Dice}_c.
$$
Dice 直接刻画预测区域与真实区域的重叠程度，对类别不平衡与小目标区域通常更敏感。

**(3) CE + Dice 组合损失**

CE与Dice的组合损失一般写为
$$
\mathcal{L}_{\mathrm{mix}}=\lambda_{\mathrm{CE}}\mathcal{L}_{\mathrm{CE}}+\lambda_{\mathrm{Dice}}\mathcal{L}_{\mathrm{Dice}},
$$
其中 $\lambda_{\mathrm{CE}},\lambda_{\mathrm{Dice}}\ge 0$ 为权重系数。若使用凸组合写法，可记为
$$
\mathcal{L}_{\mathrm{mix}}=\alpha\mathcal{L}_{\mathrm{CE}}+(1-\alpha)\mathcal{L}_{\mathrm{Dice}},\quad \alpha\in[0,1].
$$
本实验中组合损失采用等权设置，也即 $\lambda_{\mathrm{CE}}=1,\lambda_{\mathrm{Dice}}=1$；等价地在凸组合记法下可视作 $\alpha=0.5$，仅相差一个整体常数尺度。

#### 3.3. mIoU 表现对比

下面分别为三种损失函数下，第一轮( `30 epoch `)与第二轮( `50 epoch + patience = 10 `)的运行结果。

**3.3.表1. 30 epoch 对比**

| 损失函数类型 | 最佳验证集mIoU | 测试集mIoU | 最佳 epoch |
|------|------|------|------|
| CE | 0.7438 | 0.7441 | 30 |
| Dice | **0.7559** | **0.7613** | 30 |
| CE + Dice | 0.7532 | 0.7573 | 30 |

**3.3.表2. 50 epoch + patience=10 对比**

| 损失函数类型 | 最佳验证集mIoU | 测试集mIoU | 最佳 epoch |
|------|------|------|------|
| CE | 0.7559 | 0.7574 | 41 |
| Dice | **0.7716** | **0.7731** | 47 |
| CE + Dice | 0.7631 | 0.7641 | 42 |

从两张对比表出发，可以看到一个一致现象：在将训练设置从 `30 epoch` 扩展为 `50 epoch + patience=10` 后，三种损失函数在验证集与测试集上的 mIoU 均出现整体提升。这说明在当前任务与模型容量下，增加训练预算并配合早停能够带来更充分的收敛，同时避免训练后期无效迭代。

进一步结合 benchmark 本身的设定分析，Oxford-IIIT Pet 的 trimap 属于三分类像素任务（前景/边界/背景），其中**背景像素占比通常最高**，而**边界类别区域窄、像素数少且形态复杂**。这会带来两个直接后果：第一，按像素平均的 CE 优化更容易被大量易分类背景像素主导；第二，评估指标 mIoU 对各类别做等权平均，边界类一旦预测不佳会显著拖低总体分数。Dice 以类别重叠度为核心，并在分母中引入预测与真实区域规模归一化，天然弱化了多数类像素数量对优化方向的支配，更能把训练注意力拉回到前景轮廓与细边界，因此在该 benchmark 下与 mIoU 的目标一致性更高。

在此基础上再看同一轮内的损失函数排序：Dice 在两轮实验中均稳定取得最高验证 mIoU 与最高测试 mIoU；CE+Dice 虽然稳定优于仅 CE，但仍低于纯 Dice。其原因并非组合无效，而是当前等权组合
$$
\mathcal{L}_{\mathrm{mix}}=\mathcal{L}_{\mathrm{CE}}+\mathcal{L}_{\mathrm{Dice}}
$$
同时引入了两种优化偏好：CE 倾向提升逐像素分类置信度（尤其在多数背景像素上贡献更大），Dice 倾向直接提升区域重叠。对本任务最关键的**少量边界+结构连续性**这一特性而言，这两种梯度信号并不总是同向；当 CE 分量相对更强时，模型会把一部分更新预算用于改善已较容易的背景分类，而不是继续压缩边界误差，从而导致组合损失虽较 CE 有增益，却难以超过纯 Dice。

下图为训练轮次30代时三种不同损失配置训练模型的训练集与验证集loss与mIoU曲线。

![](task3_segmentation/ReportFig/3_3_ep30.png)

下图为训练轮次50代时三种不同损失配置训练模型的训练集与验证集loss与mIoU曲线。

![](task3_segmentation/ReportFig/3_3_ep50.png)

从两张曲线图中可以更直观看到三种损失在**收敛速度、后期增益和泛化稳定性**上的差异。`30 epoch` 图中，三组实验都在前 10 个 epoch 内快速收敛，但 CE 的验证曲线在早期波动更明显（验证 loss 与验证 mIoU 都出现过短暂反复）；Dice 与 CE+Dice 的验证曲线整体更平滑。到第 30 轮末，Dice 与 CE+Dice 的验证 mIoU 仍在缓慢上升，说明该训练预算下仍有继续优化空间，这与表 1 中三组最佳 epoch 都落在末段的现象一致。

`50 epoch + patience=10` 图进一步拉开了后期差距：Dice 在中后期仍保持稳定下降的验证 loss 和持续上升的验证 mIoU，最终达到最高平台；CE+Dice 的曲线形态居中，后期仍有提升但上限低于 Dice；CE 在约 35-40 epoch 后进入更明显的平台期，验证 mIoU 的边际增益最小。换言之，延长训练轮次对三组都有帮助，但 Dice 对额外训练预算的利用效率最高。

因此，曲线证据与表格结论一致且互相补充：第一，训练从 `30 epoch` 扩展到 `50 epoch + patience=10` 能整体提升分割性能；第二，在 Oxford-IIIT Pet trimap 这类背景占优、边界类别相对稀疏且以 mIoU 为核心指标的设定下，Dice 与目标指标匹配度最高，表现为更高且更稳的验证/测试 mIoU，CE+Dice 次之，CE 最弱。

### 4. 附录：代码仓库及模型权重下载地址

**GitHub 代码仓库**（实验源码与各任务 README）：

[https://github.com/Jacky23307110248/CS50028-ComputerVision/tree/main/HW2](https://github.com/Jacky23307110248/CS50028-ComputerVision/tree/main/HW2)

（课程仓库根目录：<https://github.com/Jacky23307110248/CS50028-ComputerVision>）

**Google Drive 资源包**（数据集、Task 1/3 全量 `outputs/`、Task 2 的 `runs/` / `detectVideo/` / `weights/` 等）：

[https://drive.google.com/drive/folders/1cgVH-8hv9I9M-6XvFUm0BxcQIiJ1wesb?usp=drive_link](https://drive.google.com/drive/folders/1cgVH-8hv9I9M-6XvFUm0BxcQIiJ1wesb?usp=drive_link)

下载后请将网盘内容与 GitHub 克隆得到的 `CS50028-ComputerVision/HW2/` 目录合并；目录说明见 [`README_GDrive.md`](README_GDrive.md)。